# A3 Probability Theory

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE)
[![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


### Concept Overview: Modular Functions
**Description:** Self-contained blocks of code designed to perform a specific task, accepting inputs and returning outputs.
**Economic Context:** Functions map directly to mathematical operators (e.g., utility functions, production functions) and promote code reusability.
**Key Insight:** Modular design allows for easier debugging and unit testing of economic models.



### Concept Overview: Iterative Control Flow
**Description:** A control structure that repeats a block of code for a sequence of elements.
**Economic Context:** Used for simulating time-series dynamics ($t=0, ..., T$) or iterating through a population of agents.
**Key Insight:** Explicit loops can be computationally expensive in Python; vectorized operations are preferred for large datasets.



### Concept Overview: Linear Algebra Solvers
**Description:** Computational algorithms for finding the solution vector $x$ in a system of linear equations $Ax = b$.
**Economic Context:** Used in input-output models, linearized DSGE models, and finite element methods.
**Key Insight:** Direct solvers (`np.linalg.solve`) are numerically more stable and efficient than computing the matrix inverse.



### Implementation Detail
The following code block implements the logic described above. A step-by-step breakdown follows:


In [ ]:
# === Environment Setup ===
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, linalg
from scipy.optimize import fsolve
import seaborn as sns
from IPython.display import Markdown, display

# --- Custom Display Functions ---
def theorem(title, statement):
    display(Markdown(f"""
    <div style='background-color:#e7f3e7; padding:15px; border-left:5px solid #4CAF50'>
    <strong>Theorem ({title}):</strong><br/>
    {statement}
    </div>
    """))

def example(title, content):
    display(Markdown(f"""
    <div style='background-color:#e3f2fd; padding:15px; border-left:5px solid #2196F3'>
    <strong>Example ({title}):</strong><br/>
    {content}
    </div>
    """))

def econ_app(title, content):
    display(Markdown(f"""
    <div style='background-color:#fff3e0; padding:15px; border-left:5px solid #FF9800'>
    <strong>💼 Economic Application ({title}):</strong><br/>
    {content}
    </div>
    """))

def course_connection(content):
    display(Markdown(f"""*📚 **Course Connection:** {content}*"""))

# --- Plotting Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 14, 'figure.figsize': (10, 6), 'figure.dpi': 150})
np.set_printoptions(suppress=True, linewidth=120, precision=4)

print("Environment initialized for Probability Theory.")

# The Lens: Modeling Uncertainty

**What economic problem are we solving?**
The future is unknown. Will a recession hit next year? Will this stock go up or down? Will a consumer choose product A or B? Economics is fundamentally about decision-making under uncertainty. To model this, we need a language to quantify the unknown. We cannot say *exactly* what will happen, but we can describe the *likelihood* of different outcomes.

**Why do we need this method?**
**Probability Theory** is that language. It provides the rigorous framework for defining random variables, distributions, and expectations. It allows us to process information (using Bayes' Rule), measure risk (using variance and covariance), and infer population properties from data (using the Law of Large Numbers and Central Limit Theorem). Without probability, we cannot have econometrics, finance, or modern macroeconomics.

## Table of Contents

- [3.1 Probability Spaces](#3.1-Probability-Spaces)
- [3.2 Expectation and Moments](#3.2-Expectation-and-Moments)
- [3.3 Conditional Expectation](#3.3-Conditional-Expectation)
- [3.4 Convergence of Random Variables](#3.4-Convergence-of-Random-Variables)
- [3.5 Common Distributions](#3.5-Common-Distributions)
- [3.6 Introduction to Martingales](#3.6-Introduction-to-Martingales)
- [Summary](#Summary)

## 3.1 Probability Spaces

### 3.1.1 Sample Spaces, Events, and Random Variables

A **probability space** is a mathematical construct that models a random process. It consists of three parts: $(\\Omega, \\mathcal{F}, P)$.
- **Sample Space ($\Omega$)**: The set of all possible outcomes.
- **Sigma-Algebra ($\mathcal{F}$)**: A collection of subsets of $\Omega$ called **events**. This collection is closed under complement and countable unions.
- **Probability Measure ($P$)**: A function that assigns a probability (a number in $[0, 1]$) to each event in $\mathcal{F}$.

A **random variable** $X$ is a function that maps outcomes from the sample space to real numbers, $X: \\Omega \\to \\mathbb{R}$.

### 3.1.2 CDF, PDF, and PMF

The behavior of a random variable is characterized by its distribution, which can be described in several ways:
- **Cumulative Distribution Function (CDF)**: $F_X(x) = P(X \le x)$. This is the most general description.
- **Probability Mass Function (PMF)** (for discrete RVs): $p_X(x) = P(X = x)$.
- **Probability Density Function (PDF)** (for continuous RVs): A function $f_X(x)$ such that $P(a \le X \le b) = \int_a^b f_X(x)dx$.

**Historical Note:** While the ideas of probability have been around for centuries, the formal axiomatic foundation of probability theory was laid by the Russian mathematician **Andrey Kolmogorov** in his 1933 book, *Foundations of the Theory of Probability*. Kolmogorov's axioms, which are based on measure theory, provide the rigorous mathematical framework for modern probability theory.

The image below shows the Probability Density Function (PDF) and the Cumulative Distribution Function (CDF) for a standard normal distribution (mean=0, standard deviation=1). The area under the PDF curve to the left of a value `x` is equal to the value of the CDF at `x`.

![Normal PDF](../images/png/normal_pdf.png)
*<center>Probability Density Function (PDF)</center>*

![Normal CDF](../images/png/normal_cdf.png)
*<center>Cumulative Distribution Function (CDF)</center>*

## 3.2 Expectation and Moments

### 3.2.1 Expected Value

The **expected value** (or mean) of a random variable is the long-run average value of repetitions of the experiment it represents. It is a weighted average of all possible values, where the weights are the probabilities.
- **Discrete:** $E[X] = \sum_x x \cdot p_X(x)$
- **Continuous:** $E[X] = \int_{-\infty}^{\infty} x \cdot f_X(x)dx$

In [ ]:
theorem("Law of the Unconscious Statistician (LOTUS)", "To find the expectation of a function of a random variable, $g(X)$, one does not need to first find the distribution of $g(X)$. Instead, one can compute it directly: $E[g(X)] = \\int g(x) f_X(x)dx$.")

### 3.2.2 Variance, Covariance, and Higher Moments

- **Variance**: A measure of the spread or dispersion of a distribution. $Var(X) = E[(X - E[X])^2] = E[X^2] - (E[X])^2$.
- **Covariance**: A measure of the joint variability of two random variables. $Cov(X, Y) = E[(X - E[X])(Y - E[Y])]$.
- **Correlation**: A normalized version of covariance that is always between -1 and 1. $\rho_{XY} = \frac{Cov(X, Y)}{\sigma_X \sigma_Y}$.
- **Higher Moments**: Skewness (asymmetry) and Kurtosis (tail thickness) describe other aspects of a distribution's shape.

In [ ]:
econ_app("Portfolio Theory", "The mean and variance of asset returns are the fundamental inputs into modern portfolio theory. The covariance and correlation between different assets are crucial for understanding the benefits of diversification. A portfolio's variance depends critically on the covariance between its constituent assets."),
course_connection("These concepts are foundational for all of econometrics (Chapter 6) and finance (Chapter 9).")

### 3.2.3 Jensen's Inequality

In [ ]:
theorem("Jensen's Inequality", "If $X$ is a random variable and $g$ is a convex function, then $g(E[X]) \\le E[g(X)]$. If $g$ is a concave function, the inequality is reversed: $g(E[X]) \\ge E[g(X)]$. The inequality is strict if $g$ is strictly convex/concave and $X$ is not a constant.")

**Motivation**: Jensen's Inequality provides the mathematical foundation for the economic theory of risk aversion. It establishes a direct link between the **curvature** of an agent's utility function and their preference for risk. For a concave (i.e., 'hill-shaped') utility function, the inequality shows that the utility of a certain outcome is always higher than the expected utility of a risky gamble with the same expected value. This formalizes the idea that a risk-averse individual prefers a sure thing over a gamble. The gap between the utility of the expected value and the expected utility of the gamble is a measure of the cost of risk, which gives rise to concepts like the risk premium and insurance.


### Concept Overview: Utility Theory
**Description:** A representation of preferences over a set of goods and services.
**Economic Context:** The objective function in consumer theory; functional forms like CRRA determine risk aversion and intertemporal substitution.
**Key Insight:** The properties of the utility function (e.g., curvature) drive the economic behavior of the agent.


In [ ]:
econ_app("Risk Aversion", "An agent is risk-averse if they have a concave utility function, $U$. Consider a lottery (a random variable $X$) that pays $x_1$ or $x_2$ with equal probability. The expected wealth is $E[X]$. The utility of the expected wealth is $U(E[X])$. The expected utility of the lottery is $E[U(X)]$. By Jensen's Inequality, because $U$ is concave, we have $U(E[X]) > E[U(X)]$. This means a risk-averse agent prefers the certain outcome (the expected value of the lottery) to the risky lottery itself. The difference between $E[X]$ and the 'certainty equivalent' (the amount of certain wealth that gives the same utility as the lottery) is the risk premium.")

The plot below illustrates Jensen's Inequality for a concave utility function (like $U(x)=\\log(x)$). The expected utility from a gamble over two outcomes, $E[U(X)]$, is lower than the utility of the expected value of that gamble, $U(E[X])$.

![Jensen's Inequality](../images/png/jensen_inequality.png)
*<center>This image was programmatically generated for the course.</center>*

## 3.3 Conditional Expectation

### 3.3.1 Conditional Expectation and Bayes' Rule

**Conditional expectation**, $E[Y|X]$, is the expected value of a random variable $Y$, given that another random variable $X$ has taken on a specific value. It is the best prediction of $Y$ given the information in $X$. A key tool for understanding this is **Bayes' Rule**, which allows us to update our beliefs (probabilities) in light of new evidence:
$$ P(A|B) = \frac{P(B|A)P(A)}{P(B)} $$

### 3.3.2 The Law of Iterated Expectations (LIE)

In [ ]:
theorem("Law of Iterated Expectations", "The expected value of the conditional expectation of $Y$ given $X$ is the same as the unconditional expectation of $Y$. Formally, $E[E[Y|X]] = E[Y]$. The smaller sigma-algebra always wins.")

**Motivation**: The LIE is a powerful tool for breaking down complex expectations into simpler, nested ones. It is used extensively in **rational expectations models**, where agents' expectations of future variables are formed based on today's information. The law allows us to relate expectations across different time periods and information sets in a consistent way. For example, our best guess today about what our best guess will be tomorrow is simply our best guess today. In econometrics, it is the foundation for the exogeneity assumption in OLS. The assumption that the error term has a mean of zero *conditional on the regressors*, $E[u|X] = 0$, is a statement about the conditional expectation. The LIE tells us that if this holds, the unconditional expectation of the error must also be zero: $E[u] = E[E[u|X]] = E[0] = 0$.


### Implementation Detail
The following code block implements the logic described above. A step-by-step breakdown follows:


In [ ]:
# Python Demo: Law of Iterated Expectations
np.random.seed(42)
n = 10000
X = np.random.choice([0, 1], p=[0.5, 0.5], size=n)
# Y depends on X
Y = np.zeros(n)
Y[X==0] = np.random.normal(5, 1, size=np.sum(X==0))
Y[X==1] = np.random.normal(10, 1, size=np.sum(X==1))

E_Y = np.mean(Y)
E_Y_given_X0 = np.mean(Y[X==0])
E_Y_given_X1 = np.mean(Y[X==1])

# E[E[Y|X]] = P(X=0)*E[Y|X=0] + P(X=1)*E[Y|X=1]
E_E_Y_X = 0.5 * E_Y_given_X0 + 0.5 * E_Y_given_X1

print(f"E[Y]: {E_Y:.4f}")
print(f"E[E[Y|X]]: {E_E_Y_X:.4f}")

In [ ]:
course_connection("Underpins all of econometrics (Chapter 6) and rational expectations models (Chapter 4).")

## 3.4 Convergence of Random Variables

### 3.4.1 Modes of Convergence

The concept of convergence for a sequence of random variables is more nuanced than for deterministic sequences. There are several distinct 'modes' of convergence, each with a different strength and interpretation.

- **Convergence in Distribution ($X_n \xrightarrow{d} X$)**: This is the weakest form of convergence. It means that the *shape* of the cumulative distribution function (CDF) of $X_n$ gets closer and closer to the shape of the CDF of $X$. It does not mean that the random variables themselves are getting close. The Central Limit Theorem is the most famous example of convergence in distribution.
- **Convergence in Probability ($X_n \xrightarrow{p} X$)**: This is a stronger concept. It means that the probability of the random variable $X_n$ being 'far away' from the random variable $X$ becomes vanishingly small as $n$ increases. The Law of Large Numbers is a key example.
- **Almost Sure Convergence ($X_n \xrightarrow{a.s.} X$)**: This is the strongest form of convergence typically used in economics. It means that for any possible outcome of the underlying random process ($\\omega$), the sequence of realized values $X_n(\\omega)$ will converge to the value $X(\\omega)$, except for a set of 'unlucky' outcomes that has probability zero. 

**Hierarchy**: Almost sure convergence implies convergence in probability, which in turn implies convergence in distribution. The reverse implications are not generally true.

### 3.4.2 Law of Large Numbers (LLN) and Central Limit Theorem (CLT)

In [ ]:
theorem("Law of Large Numbers (LLN)", "For i.i.d. random variables $X_i$ with mean $\\mu$, the sample mean $\\bar{X}_n = \\frac{1}{n}\\sum_{i=1}^n X_i$ converges in probability to the true mean $\\mu$. The Strong LLN states that this convergence is almost sure.")

In [ ]:
theorem("Central Limit Theorem (CLT)", "For i.i.d. random variables $X_i$ with mean $\\mu$ and variance $\\sigma^2$, the standardized sample mean converges in distribution to a standard normal distribution: $ \\frac{\\bar{X}_n - \\mu}{\\sigma/\\sqrt{n}} \\xrightarrow{d} N(0, 1) $. ")

**Why they matter**: The LLN justifies using sample averages to estimate population means. The CLT is the reason the normal distribution is ubiquitous in statistics; it allows us to perform hypothesis testing and construct confidence intervals for sample means, even if the underlying data is not normally distributed.

The images below provide a simple illustration of the Central Limit Theorem. We start with a discrete uniform distribution (a single die roll). As we sum more and more independent draws from this distribution, the distribution of the sum begins to look more and more like a normal distribution.

![CLT with 1 die](../images/png/clt_1.png)
*<center>Distribution for the sum of 1 die.</center>*

![CLT with 2 dice](../images/png/clt_2.png)
*<center>Distribution for the sum of 2 dice.</center>*

![CLT with 3 dice](../images/png/clt_3.png)
*<center>Distribution for the sum of 3 dice. Note the emerging bell shape.</center>*

In [ ]:
course_connection("The LLN and CLT are the foundation of all statistical inference in Chapter 6.")

## 3.5 Common Distributions

### 3.5.1 Key Probability Distributions

A brief overview of the most important distributions for economics and econometrics:

**Discrete:**
- **Bernoulli**: A single trial with two outcomes (e.g., success/failure).
- **Binomial**: The number of successes in a fixed number of independent Bernoulli trials.
- **Poisson**: The number of events occurring in a fixed interval of time or space.

**Continuous:**
- **Uniform**: All outcomes in a range are equally likely.
- **Normal (Gaussian)**: The classic bell curve, central to statistics due to the CLT.
- **Lognormal**: The logarithm of the random variable is normally distributed. Often used for variables that cannot be negative, like income or stock prices.
- **Chi-squared (χ²)**: The distribution of a sum of squared standard normal variables. Used in hypothesis testing.
- **Student's t**: Similar to the normal but with heavier tails. Used for inference when the sample size is small.
- **F-distribution**: The distribution of a ratio of two chi-squared variables. Used for F-tests in regression.

### 3.5.2 The Multivariate Normal Distribution

The **multivariate normal distribution** is the generalization of the normal distribution to higher dimensions. A vector $\\mathbf{X} \\in \\mathbb{R}^n$ is multivariate normal, $\\mathbf{X} \\sim N(\\oldsymbol{\\mu}, \\Sigma)$, if it is characterized by a mean vector $\\boldsymbol{\\mu}$ and a covariance matrix $\\Sigma$. It has several crucial properties:
- Linear combinations of its components are normally distributed.
- All marginal distributions are normal.
- All conditional distributions are normal.
- Zero correlation implies independence (this is a special property not true for most distributions).

The image below shows samples drawn from three different bivariate normal distributions. The correlation coefficient $\rho$ determines the orientation and shape of the distribution. When $\rho$ is positive, the variables are positively related; when $\rho$ is negative, they are negatively related; and when $\rho$ is zero, they are uncorrelated and, in the case of the normal distribution, independent.

![Bivariate Normal Distributions](../images/png/multivariate_normal_distribution.png)
*<center>This image was programmatically generated for the course.</center>*

## 3.6 Introduction to Martingales

### 3.6.1 Definition and Intuition

A **martingale** is a stochastic process (a sequence of random variables) for which, at any particular time, the conditional expectation of the next value in the sequence, given all prior values, is equal to the present value. Informally, it's a model of a **fair game**; if you know the history of the game up to today, your best bet for the value of your fortune tomorrow is its value today.

In [ ]:
theorem("Martingale Property", "A stochastic process $\{M_t\}$ is a martingale with respect to an information set (filtration) $\\mathcal{F}_t$ if it satisfies:<br>1. $E[|M_t|] < \\infty$ for all $t$.<br>2. $E[M_{t+1} | \\mathcal{F}_t] = M_t$."),
econ_app("Efficient Markets Hypothesis", "A simple version of the EMH states that asset prices should follow a martingale. If a price change were predictable, investors would trade on that information until the predictable component was eliminated. Therefore, under the EMH, the best forecast of tomorrow's stock price is today's stock price.")

The plot below shows a simulation of several paths of a random walk, which is a classic example of a martingale. At any point in time, the expected next position is the current position.

![Martingale Paths](../images/png/martingale_paths.png)
*<center>This image was programmatically generated for the course.</center>*

In [ ]:
course_connection("Crucial for asset pricing (Chapter 9), rational expectations models (Chapter 4), and the theory of time series econometrics (Chapter 8).")

---
## Summary

In this lecture, we have systematically explored the theoretical and practical aspects of the model.

**Key Takeaways:**
1.  **Foundations:** We established the mathematical basis of the economic problem.
2.  **Computation:** We implemented the solution using efficient algorithms.
3.  **Implications:** We analyzed the economic significance of the results.

**Further Exploration:**
- Experiment with model parameters to assess sensitivity.
- Extend the framework by relaxing simplifying assumptions.